<h3 style="color:teal;text-align:center">⌞Model training⌟</h3>

<h4>• XGBoost benchmark</h4>

In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# 1. Load the finalized dataset
df = pd.read_csv("dataset/aquair_final.csv", parse_dates=["timestamp(UTC+1)"])
df.set_index("timestamp(UTC+1)", inplace=True)

# 2. Define Features (X) and Target (y)
features = [c for c in df.columns if any(k in c for k in ["lag", "sin", "cos", "rolling", "inter"])]
X = df[features]
y = df["target_pm25_15m"]

# 3. Chronological Split (80% Train, 20% Test)
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# 4. Time-series CV + randomized search
tscv = TimeSeriesSplit(n_splits=5)

xgb = XGBRegressor(
    random_state=42,
    tree_method="hist",
    n_jobs=-1
)

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

xgb_params = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

rf_params = {
    "n_estimators": [200, 400, 600],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.7]
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_params,
    n_iter=25,
    scoring="neg_mean_squared_error",
    cv=tscv,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

rf_search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=25,
    scoring="neg_mean_squared_error",
    cv=tscv,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

print(f"Training on {len(X_train)} rows from Azrou facility...")
models = {"XGBoost": xgb_search, "Random Forest": rf_search}
results = {}

for name, search in models.items():
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    preds = best_model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
        "BestParams": search.best_params_
    }
    print(f"[{name}] Finished training.")

print("\n--- Model Performance Benchmarking ---")
for name, m in results.items():
    print(f"{name}: R2 = {m['R2']:.4f}, MAE = {m['MAE']:.2f}, RMSE = {m['RMSE']:.2f}")
    print(f"  Best params: {m['BestParams']}")

Training on 19051 rows from Azrou facility...
[XGBoost] Finished training.
[Random Forest] Finished training.

--- Model Performance Benchmarking ---
XGBoost: R2 = 0.9131, MAE = 1.27, RMSE = 2.35
  Best params: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
Random Forest: R2 = 0.8483, MAE = 2.56, RMSE = 3.11
  Best params: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 5}


<h4>• LightGBM benchmark</h4>

In [8]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

lgbm = lgb.LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    objective="regression",
    verbosity=-1,
    bagging_freq=1,
)

lgbm_params = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [3, 5, 7, 9],
    "bagging_fraction": [0.6, 0.8, 0.9],
    "feature_fraction": [0.7, 0.8, 0.9],
    "min_data_in_leaf": [5, 10, 20, 50],
    "lambda_l1": [0, 0.1, 1, 10],
    "lambda_l2": [0, 0.1, 1, 10],
}

lgbm_search = RandomizedSearchCV(
    lgbm,
    lgbm_params,
    n_iter=50,
    scoring="neg_mean_squared_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

lgbm_search.fit(X_train, y_train)
lgbm_best = lgbm_search.best_estimator_
lgbm_preds = lgbm_best.predict(X_test)

results["LightGBM"] = {
    "MAE": mean_absolute_error(y_test, lgbm_preds),
    "RMSE": np.sqrt(mean_squared_error(y_test, lgbm_preds)),
    "R2": r2_score(y_test, lgbm_preds),
    "BestParams": lgbm_search.best_params_,
}

print("[LightGBM] Finished training.")
print(
    f"LightGBM: R2 = {results['LightGBM']['R2']:.4f}, "
    f"MAE = {results['LightGBM']['MAE']:.2f}, "
    f"RMSE = {results['LightGBM']['RMSE']:.2f}"
)
print(f"  Best params: {results['LightGBM']['BestParams']}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] Finished training.
LightGBM: R2 = 0.7583, MAE = 2.95, RMSE = 3.92
  Best params: {'num_leaves': 255, 'n_estimators': 300, 'min_data_in_leaf': 50, 'max_depth': 9, 'learning_rate': 0.01, 'lambda_l2': 0.1, 'lambda_l1': 0, 'feature_fraction': 0.8, 'bagging_fraction': 0.8}
